In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

df = pd.read_csv('diamonds.csv')

df['cut'] =df['cut'].replace('?', np.nan)

df['cut'].mode()[0]

df['cut'].fillna(df['cut'].mode()[0],inplace=True)

df.isnull().sum()

df = sns.load_dataset('diamonds')
df.head()

df.groupby("cut", as_index=False)['price'].agg(['min', 'mean', 'max'])

df2 = df.groupby("cut", as_index=False).agg({
    'price' : ['min', 'mean', 'max'],
    })
df2

C:\Users\emili\AppData\Local\Temp\ipykernel_13804\4175320977.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['cut'].fillna(df['cut'].mode()[0],inplace=True)
C:\Users\emili\AppData\Local\Temp\ipykernel_13804\4175320977.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("cut", as_index=False)['price'].agg(['min', 'mean'

cut price                    
               min         mean    max
0      Ideal   326  3457.541970  18806
1    Premium   326  4584.257704  18823
2  Very Good   336  3981.759891  18818
3       Good   327  3928.864452  18788
4       Fair   337  4358.757764  18574

In [ ]:
import numpy as np
import pandas as pd
import plotly # Para ver la versión
import plotly.express as px
from datetime import datetime
# Versiones
print(f"numpy=={np.__version__}")
print(f"pandas=={pd.__version__}")
print(f"plotly=={plotly.__version__}")
# datasets integrados en plotly:
df = px.data.tips()
df.head()
df = px.data.iris()
df.head()
df = px.data.gapminder()
df.head()
# r'' para evitar problemas con rutas de archivos en windows, donde barras invertidas se pueden interpretar como caracteres de escape
# df = pd.read_csv(r'..\Data\Bakery sales.csv')
df = pd.read_csv('../Data/Bakery sales.csv')
df.head()
df.info()
# uso de apply para quitar ' €' y reemplazar , por . y convertir a float
def formatear_precio(price):
    numerical = price.split(' ')[0]
    numerical_point = numerical.replace(',', '.')
    return np.float32(numerical_point)
df['unit_price'].apply(formatear_precio)
df['unit_price'].str.replace(' €', '').str.replace(',','.').astype(np.float32)
df['unit_price'] = df['unit_price'].apply(lambda price: np.float32(price.split(' ')[0].replace(',', '.')))
# convertir la fecha a datetime
# datetime.strptime('2021-01-02', '%Y-%m-%d')
df['date'].apply(lambda fecha: datetime.strptime(fecha,'%Y-%m-%d'))
pd.to_datetime(df['date'])
df['date'] = df['date'].astype('datetime64[ns]')
df.info()
# year:
df['year'] = df['date'].dt.year
df['year'].head(3)
# hour (usaremos este para facilitar los gráficos)
df['hour'] = df['time'].apply(lambda time: np.int32(time.split(':')[0]))
df['hour'].head(3)
# hour decimal
df['time'].apply(lambda time: round(int(time.split(':')[0]) + int(time.split(':')[1]) / 60, 2))
fig = px.histogram(df, x='hour', y='Quantity', color='year', nbins=48)
fig.update_layout(xaxis= {
    'range': [0, 24],
    'tickvals': list(range(1, 25))
})
## 11 am es cuando más productos se venden
fig = px.histogram(df, x='hour', y='Quantity', facet_col='year', nbins=48, marginal='rug')
fig.update_layout(xaxis= {
    'range': [0, 24],
    'tickvals': list(range(1, 25))
}, yaxis= {
    'range': [0, 40_000]
})
# fig.update_layout(xaxis_matches='x', yaxis_matches='y')
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(range=[0, 24] if axis.startswith('xaxis') else [0, 40000])
fig.show()
df_baguette = df[df['article'].str.contains('BAGUETTE')]
px.sunburst(df_baguette, values='Quantity', path=['year', 'month'])
# otro ejemplo:
df_countries = px.data.gapminder().query("year == 2007")
fig = px.sunburst(df_countries, path=['continent', 'country'], values='pop',
                  color='lifeExp', hover_data=['iso_alpha'],
                  color_continuous_scale='RdBu',
                  color_continuous_midpoint=np.average(df_countries['lifeExp'], weights=df_countries['pop']))
fig.show()
df["total_article_price"] = df["Quantity"] * df["unit_price"]
df.head(4)
Q1 = df['Quantity'].quantile(0.25)
Q1
Q3 = df['Quantity'].quantile(0.75)
Q3
IQR = Q3 - Q1
IQR
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
filtro_outliers = (df['Quantity'] <= upper_limit) & (df['Quantity'] >= lower_limit)
df_no_outliers = df[filtro_outliers]
# violin plot
px.violin(df, x='total_article_price')
px.violin(df_no_outliers, x='total_article_price')
df.groupby('date')['total_article_price'].sum().plot()
df_sales = df.groupby('date', as_index=False).agg({
    'total_article_price': 'sum',
    'weekday': 'min'
})
df_sales.head(3)
px.line(df_sales, x='date', y='total_article_price', color='weekday')
df.head(4)
df_ticket = df_no_outliers.groupby("ticket_number", as_index=False).agg(
    {"Quantity": "sum", "total_article_price": "sum"}
)
df_ticket.head(4)
# Gráfica sin transformación logarítmica
px.box(df_ticket, x='Quantity', y='total_article_price')
px.histogram(df_ticket, x='total_article_price')
# Transformación logarítmica:
# si tiene valores 0 o negativos le puede afectar, conviene filtrarlos primero
df_ticket = df_ticket[df_ticket['total_article_price'] > 0]
df_ticket['price_log'] = df_ticket['total_article_price'].apply(np.log)
df_ticket.head(4)
# Gráfica sin transformación logarítmica
px.box(df_ticket, x='Quantity', y='total_article_price')
px.histogram(df_ticket, x='total_article_price')
# Transformación logarítmica:
# si tiene valores 0 o negativos le puede afectar, conviene filtrarlos primero
df_ticket = df_ticket[df_ticket['total_article_price'] > 0]
df_ticket['price_log'] = df_ticket['total_article_price'].apply(np.log)
df_ticket.head(4)
px.histogram(df_ticket, x='price_log')
np.exp([-1 , 0, 1, 2, 3, 4, 5])
fig = px.box(df_ticket, x='Quantity', y='price_log')
fig.update_layout(
    yaxis= {
        'range': [-1 , 5],
        'tickvals': [-1, 0, 1, 2, 3, 4, 5],
        'ticktext': [f'{x:.2f} €' for x in np.exp([-1 , 0, 1, 2, 3, 4, 5])]
    }
)
# si no ponemos np.exp las etiquetas ticktext se mantendrán en la escala logarítmica: -1, 0, 1, 2, 3, 4, 5
#np.exp revertir la transformación logarítmica para regresar a la escala original

# Módulo 3: Data science, ejercício resuelto

## 1: Carga y limpieza de datos:

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px

# Carga del dataset
df = pd.read_csv('diamonds.csv')

# Reemplazar valores '?' por NaN
df.replace('?', np.nan, inplace=True)

# Cambio de tipo de datos
df['carat'] = df['carat'].astype(np.float32)
df['cut'] = df['cut'].astype('category')

# Limpieza de valores nulos
# Columnas continuas: reemplazo por la mediana
df['depth'] = df['depth'].fillna(df['depth'].median())
df['table'] = df['table'].fillna(df['table'].median())

# Columnas categóricas: reemplazo por la moda
df['cut'] = df['cut'].fillna(df['cut'].mode()[0])

# Encoding
# One-hot encoding para variables categóricas
df_one_hot = pd.get_dummies(df, columns=['color', 'clarity'], drop_first=True)

# Ordinal encoding para la columna "cut"
cut_mapping = {'Fair': 1, 'Good': 2, 'Very Good': 3, 'Premium': 4, 'Ideal': 5}
df['cut_int'] = df['cut'].map(cut_mapping)

## 2: Transformaciones

In [13]:
# Columna price_iva (precio con IVA del 21%)
df['price_iva'] = df['price'] * 1.21

# Columna price_discount (precio con descuentos aplicados)
def calculate_discount(price, cut):
    if price < 1000 and cut == 'Ideal':
        return price * 0.9
    elif 1000 <= price <= 5000 and cut == 'Premium':
        return price * 0.85
    else:
        return price

df['price_discount'] = df.apply(lambda row: calculate_discount(row['price'], row['cut']), axis=1)

# Columna volumen (x * y * z)
df['volume'] = df['x'] * df['y'] * df['z']

# Ordenar por cut y price
df_sorted = df.sort_values(by=['cut', 'price'], ascending=[True, True])

# Agrupaciones
grouped = df.groupby(['cut', 'color', 'clarity']).agg({
    'price': ['mean', 'max', 'min'],
    'carat': ['mean', 'max', 'min']
}).reset_index()

TypeError: can't multiply sequence by non-int of type 'float'

## 3: Distribuciones

In [ ]:
# Visualización de outliers (columna price)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Filtro de outliers
filtro_outliers = (df['price'] >= lower_limit) & (df['price'] <= upper_limit)
df_no_outliers = df[filtro_outliers]

# Asimetría, curtosis y transformación logarítmica
df['price_log'] = np.log(df['price'] + 1)  # Evita problemas con valores 0 o negativos

# Histograma y boxplot por tipo de corte
px.histogram(df, x='price_log', color='cut')
px.box(df, x='cut', y='price_log')

# Discretización de precios
bins = [0, 1000, 5000, df['price'].max()]
labels = ['Barato', 'Medio', 'Caro']
df['price_category'] = pd.cut(df['price'], bins=bins, labels=labels)

## 4: Visualizaciones

In [ ]:
# Univariantes: Histogramas y curvas de densidad
sns.histplot(df['carat'], kde=True)
sns.kdeplot(df['price'], shade=True)

# Boxplot
sns.boxplot(x='cut', y='price', data=df)

# Countplot
sns.countplot(x='cut', data=df)

# Bivariantes y multivariantes: Scatterplot
sns.scatterplot(x='carat', y='price', hue='cut', style='color', size='depth', data=df)

# Correlación con Pandas y visualización con Seaborn
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm')

# Gráfico de barras para la columna 'price'
correlation_price = correlation['price'].sort_values(ascending=False)
correlation_price.plot(kind='bar')

# Pivot table y heatmap
pivot = df.pivot_table(index='cut', columns='color', values='price', aggfunc='mean')
sns.heatmap(pivot, annot=True, cmap='viridis')

# Gráficas combinadas usando relplot
sns.relplot(x='carat', y='price', hue='cut', col='color', kind='scatter', data=df)


## Visualización de gráficos con Plotly

In [ ]:
# Histograma con Plotly
px.histogram(df, x='price', color='cut', nbins=50)

# Gráfico de dispersión 3D
px.scatter_3d(df, x='carat', y='price', z='depth', color='cut')

# Gráfico de líneas para precios agrupados por fecha simulada
df['date'] = pd.date_range(start='2023-01-01', periods=len(df))
df_sales = df.groupby('date', as_index=False)['price'].sum()
px.line(df_sales, x='date', y='price')
